# Neuromelanin pipeline — Jupyter orchestrating headless QuPath

Everything runs inside **real QuPath** (Cellpose2D extension, your saved object classifier, your NM measurements). This notebook just generates the Groovy scripts, launches QuPath from the command line (no GUI), and handles everything around it: results, summaries, training-data export, and classifier metrics.

**Run order**
1. Config → locate project / classifier
2. Generate scripts
3. Run the full pipeline (detect → NM features → classify → export)
4. Collect NM measurements + slide summary
5. Training metrics (export training cells, evaluate)

**Retraining** is done in the QuPath GUI (see the last section); afterwards you re-run sections 3–5 here.

## 1. Configuration

In [1]:
import subprocess, json, re, shutil, hashlib, time
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- paths (edit these) ----
# macOS:   /Applications/QuPath.app/Contents/MacOS/QuPath
# Windows: use the *console* launcher, e.g. C:\Program Files\QuPath-0.5.1\QuPath-0.5.1 (console).exe
QUPATH_EXE        = "/Applications/QuPath-0.7.0-arm64.app"
PROJECT_PATH      = "/Users/seymour/Downloads/Shomaiya/Neuromelanin/qupath/project.qpproj"
CLASSIFIER_NAME   = "Pigmented_Neurons"      # name of the saved object classifier in the project
CLASSIFIER_SOURCE = None                      # optional: path to a Pigmented_Neurons.json to copy into the project

# ---- which slides ----
IMAGE_FILTER = None      # None = every image in the project, or a list like ["slide_001.svs", "slide_002.svs"]

# ---- pipeline parameters (same values as your existing script) ----
CELLPOSE_MODEL      = "cyto3"
PIXEL_SIZE_UM       = 0.5
CELL_DIAMETER       = 95
CELL_EXPANSION      = 0
NM_RGB_THRESHOLD    = 100    # pixels with mean RGB below this count as pigmented
TIMEOUT_SEC         = 3 * 3600

# ---- classes used in training / results ----
POS_CLASS = "Pigmented Neuron"
NEG_CLASS = "Negative"

## 2. Locate project, slides and classifier

Reads the slide list straight from the `.qpproj` file, and reads the classifier's required feature list from its JSON, so the notebook, the Groovy scripts and the metrics all use the same 21 features.

In [2]:
PROJECT_PATH = Path(PROJECT_PATH)
project_dir  = PROJECT_PATH.parent
proj         = json.loads(PROJECT_PATH.read_text())
all_images   = [e["imageName"] for e in proj["images"]]
IMAGE_NAMES  = all_images if IMAGE_FILTER is None else [n for n in all_images if n in IMAGE_FILTER]
print(f"{len(IMAGE_NAMES)} of {len(all_images)} project images selected")

classifier_dir  = project_dir / "classifiers" / "object_classifiers"
classifier_path = classifier_dir / f"{CLASSIFIER_NAME}.json"
if CLASSIFIER_SOURCE:
    classifier_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(CLASSIFIER_SOURCE, classifier_path)
assert classifier_path.exists(), f"Classifier not found: {classifier_path}"

clf_json         = json.loads(classifier_path.read_text())
FEATURE_NAMES    = clf_json["featureExtractor"]["measurements"]
CLASSIFIER_CLASSES = [c["name"] for c in clf_json["pathClasses"]]
print("Classifier classes:", CLASSIFIER_CLASSES)
print(f"{len(FEATURE_NAMES)} features:", FEATURE_NAMES)

# output folders (all inside the project folder, same place your script already used)
scripts_dir  = project_dir / "/Users/seymour/Downloads/Shomaiya/Neuromelanin/qupath/scripts"
logs_dir     = project_dir / "/Users/seymour/Downloads/Shomaiya/Neuromelanin/qupath/pipeline_logs"
nm_dir       = project_dir / "/Users/seymour/Downloads/Shomaiya/Neuromelanin/qupath/neuromelanin_measurements"
train_dir    = project_dir / "/Users/seymour/Downloads/Shomaiya/Neuromelanin/qupath/training_data"
metrics_dir  = project_dir / "/Users/seymour/Downloads/Shomaiya/Neuromelanin/qupath/training_metrics"
versions_dir = project_dir / "/Users/seymour/Downloads/Shomaiya/Neuromelanin/qupath/classifier_versions"
for d in (scripts_dir, logs_dir, nm_dir, train_dir, metrics_dir, versions_dir):
    d.mkdir(exist_ok=True)

def safe_name(s): return re.sub(r"[^a-zA-Z0-9.-]", "_", s)

37 of 37 project images selected
Classifier classes: ['Negative', 'Pigmented Neuron']
21 features: ['Circularity', 'Solidity', 'Hematoxylin: Mean', 'Hematoxylin: Median', 'Hematoxylin: Min', 'Hematoxylin: Max', 'Hematoxylin: Std.Dev.', 'Eosin: Mean', 'Eosin: Median', 'Eosin: Min', 'Eosin: Max', 'Eosin: Std.Dev.', 'NM: Cell area um2', 'NM: Pigmented area um2', 'NM: Pigmented area fraction', 'NM: Mean cell OD', 'NM: Mean pigmented OD', 'NM: Integrated Density', 'NM: Integrated pigmented OD', 'NM: Pigment OD std dev', 'NM: Peak pigment OD']


## 3. Generate the Groovy scripts

**Important change vs. your current script:** your classifier uses the `NM:` measurements as features, but the old script classified *first* and only computed `NM:` values for cells that were already classified as pigmented. The classifier therefore never saw NM features for negative cells (and saw none at all on a first run). Here, NM measurements are computed for **all** detections *before* classification, and a preflight check refuses to classify if any of the classifier's required features are missing, rather than silently classifying on blanks.

In [3]:
PIPELINE_TEMPLATE = r"""
import qupath.ext.biop.cellpose.Cellpose2D
import qupath.lib.regions.RegionRequest
import qupath.lib.roi.RoiTools
import qupath.lib.roi.ROIs
import qupath.lib.objects.PathObjects
import java.awt.image.BufferedImage
import java.awt.geom.Area
import groovy.json.JsonSlurper

// ---------- settings injected by the notebook ----------
def CLASSIFIER_NAME = "__CLASSIFIER_NAME__"
def CELLPOSE_MODEL  = "__CELLPOSE_MODEL__"
double PIXEL_SIZE   = __PIXEL_SIZE__
double DIAMETER     = __DIAMETER__
double EXPANSION    = __EXPANSION__
double RGB_THRESHOLD = __RGB_THRESHOLD__
boolean RUN_CLASSIFIER = __RUN_CLASSIFIER__
def POS_CLASS_NAME  = "__POS_CLASS__"

def imageData = getCurrentImageData()
def server = imageData.getServer()
def imageName = server.getMetadata().getName()
def entry = getProjectEntry()

// ---------- 1. Cellpose detection (skipped if detections already exist) ----------
def existingDetections = getDetectionObjects()
if (!existingDetections.isEmpty()) {
    println "Found " + existingDetections.size() + " existing detections on " + imageName + " - skipping Cellpose."
} else {
    println ("No detections on " + imageName + " - running Cellpose " + CELLPOSE_MODEL + "...")
    
    def lineObjects = getAnnotationObjects().findAll { it.getROI().isLine() }
    for (lineObject in lineObjects) {
        def line = lineObject.getROI()
        def points = line.getAllPoints()
        if (points.size() <= 2) continue
        def polygon = ROIs.createPolygonROI(points, line.getImagePlane())
        def polygonObject = PathObjects.createAnnotationObject(polygon, lineObject.getPathClass())
        polygonObject.setName(lineObject.getName())
        polygonObject.setColor(lineObject.getColor())
        addObject(polygonObject)
        removeObject(lineObject, true)
    }
    fireHierarchyUpdate()

    def cellpose = Cellpose2D.builder(CELLPOSE_MODEL)
            .pixelSize(PIXEL_SIZE)
            .channels('Red', 'Green', 'Blue')
            .diameter(DIAMETER)
            .cellExpansion(EXPANSION)
            .measureShape()
            .measureIntensity()
            .build()

    def parentObjects = getAnnotationObjects()
    if (parentObjects.isEmpty()) {
        println "SKIPPED (no annotations found): " + imageName
        println "PIPELINE_SKIPPED " + imageName
        return
    }
    cellpose.detectObjects(imageData, parentObjects)
    println "Cellpose detection complete."
    if (entry != null) entry.saveImageData(imageData)
}

// ---------- 2. Neuromelanin measurements for ALL detections (before classification) ----------
def cal = server.getPixelCalibration()
double pxArea = cal.getPixelWidthMicrons() * cal.getPixelHeightMicrons()

def allCells = getDetectionObjects()
println "Computing NM measurements for " + allCells.size() + " detections..."
int processed = 0
allCells.each { cell ->
    def roi = cell.getROI()
    def request = RegionRequest.createInstance(server.getPath(), 1, roi)
    BufferedImage img = server.readRegion(request)
    def raster = img.getRaster()
    def area = new Area(RoiTools.getShape(roi))

    int totalPixels = 0
    int pigmentedPixels = 0
    double sumOD = 0
    double sumODsq = 0
    double sumODall = 0
    double minRGB = 255

    int w = img.getWidth()
    int h = img.getHeight()
    int offX = request.getX()
    int offY = request.getY()

    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            if (!area.contains(x + offX, y + offY)) continue
            totalPixels++
            int r = raster.getSample(x, y, 0)
            int g = raster.getSample(x, y, 1)
            int b = raster.getSample(x, y, 2)
            double meanRGB = (r + g + b) / 3.0
            if (meanRGB < minRGB) minRGB = meanRGB
            double od = -Math.log10(Math.max(meanRGB, 1) / 255.0)
            sumODall += od
            if (meanRGB < RGB_THRESHOLD) {
                pigmentedPixels++
                sumOD += od
                sumODsq += od * od
            }
        }
    }
    if (totalPixels == 0) return

    double cellAreaUm2       = totalPixels * pxArea
    double pigmentedAreaUm2  = pigmentedPixels * pxArea
    double pigmentedFraction = pigmentedPixels / (double) totalPixels
    double meanPigmentedOD   = pigmentedPixels > 0 ? sumOD / pigmentedPixels : 0
    double meanCellOD        = sumODall / totalPixels
    double integratedDensity = meanCellOD * cellAreaUm2
    double integratedPigOD   = sumOD * pxArea
    double odVar             = pigmentedPixels > 1 ? (sumODsq / pigmentedPixels) - (meanPigmentedOD * meanPigmentedOD) : 0
    double odStdDev          = odVar > 0 ? Math.sqrt(odVar) : 0
    double peakOD            = -Math.log10(Math.max(minRGB, 1) / 255.0)

    def ml = cell.getMeasurementList()
    ml.put("NM: Cell area um2", cellAreaUm2)
    ml.put("NM: Pigmented area um2", pigmentedAreaUm2)
    ml.put("NM: Pigmented area fraction", pigmentedFraction)
    ml.put("NM: Mean cell OD", meanCellOD)
    ml.put("NM: Mean pigmented OD", meanPigmentedOD)
    ml.put("NM: Integrated Density", integratedDensity)
    ml.put("NM: Integrated pigmented OD", integratedPigOD)
    ml.put("NM: Pigment OD std dev", odStdDev)
    ml.put("NM: Peak pigment OD", peakOD)
    ml.close()
    processed++
}
fireHierarchyUpdate()
println "NM measurements added to " + processed + " detections."
if (entry != null) entry.saveImageData(imageData)

// ---------- 3. Preflight: does every detection have every feature the classifier needs? ----------
def classifierFile = new File(buildFilePath(PROJECT_BASE_DIR, "classifiers", "object_classifiers", CLASSIFIER_NAME + ".json"))
def requiredFeatures = new JsonSlurper().parse(classifierFile).featureExtractor.measurements
def missing = [:]
getDetectionObjects().each { d ->
    def ml = d.getMeasurementList()
    requiredFeatures.each { f -> if (!ml.containsKey(f)) missing[f] = (missing[f] ?: 0) + 1 }
}
if (!missing.isEmpty()) {
    println "PREFLIGHT_FAILED on " + imageName + " - detections are missing classifier features:"
    missing.each { f, n -> println "   " + f + " (missing on " + n + " detections)" }
    return
}
println "Preflight OK: all " + requiredFeatures.size() + " classifier features present."

// ---------- 4. Classify ----------
if (RUN_CLASSIFIER) {
    runObjectClassifier(CLASSIFIER_NAME)
    fireHierarchyUpdate()
    println CLASSIFIER_NAME + " classifier applied."
    if (entry != null) entry.saveImageData(imageData)
}

// ---------- 5. Counts by class ----------
def allDetections = getDetectionObjects()
def countsByClass = allDetections.countBy { it.getPathClass()?.toString() ?: "Unclassified" }
println "Detection counts for " + imageName + ": total " + allDetections.size()
countsByClass.each { k, v -> println "  " + k + ": " + v }

def countsDir = buildFilePath(PROJECT_BASE_DIR, "detection_counts")
mkdirs(countsDir)
def countsFile = new File(buildFilePath(countsDir, "detection_counts.tsv"))
def header = ["Image", "Class", "Count"].join("\t")
def keptLines = []
if (countsFile.exists()) {
    keptLines = countsFile.readLines().findAll { it != header && !it.startsWith(imageName + "\t") }
}
countsFile.withWriter { writer ->
    writer.writeLine(header)
    keptLines.each { writer.writeLine(it) }
    writer.writeLine([imageName, "Total", allDetections.size()].join("\t"))
    countsByClass.each { k, v -> writer.writeLine([imageName, k, v].join("\t")) }
}

// ---------- 6. Export per-cell NM measurements (ALL detections, with their class) ----------
def outputDir = buildFilePath(PROJECT_BASE_DIR, "neuromelanin_measurements")
mkdirs(outputDir)
def outputPath = buildFilePath(outputDir, imageName.replaceAll("[^a-zA-Z0-9.-]", "_") + "_neuromelanin.tsv")
def exportCols = ["NM: Cell area um2", "NM: Pigmented area um2", "NM: Pigmented area fraction",
                  "NM: Mean cell OD", "NM: Mean pigmented OD", "NM: Integrated Density",
                  "NM: Integrated pigmented OD", "NM: Pigment OD std dev", "NM: Peak pigment OD"]
new File(outputPath).withWriter { writer ->
    writer.writeLine((["Image", "Class", "Centroid X px", "Centroid Y px"] + exportCols).join("\t"))
    allDetections.each { cell ->
        def row = [imageName, cell.getPathClass()?.toString() ?: "Unclassified",
                   cell.getROI().getCentroidX(), cell.getROI().getCentroidY()]
        exportCols.each { col ->
            def ml = cell.getMeasurementList()
            row << (ml.containsKey(col) ? ml.get(col).toString() : "")
        }
        writer.writeLine(row.join("\t"))
    }
}
println "Exported: " + outputPath
if (entry != null) entry.saveImageData(imageData)
println "PIPELINE_OK " + imageName
"""

TRAINING_EXPORT_TEMPLATE = r"""
import groovy.json.JsonSlurper

def imageName = getCurrentServer().getMetadata().getName()
def classifierFile = new File(buildFilePath(PROJECT_BASE_DIR, "classifiers", "object_classifiers", "__CLASSIFIER_NAME__.json"))
def features = new JsonSlurper().parse(classifierFile).featureExtractor.measurements
def labelClasses = ["__POS_CLASS__", "__NEG_CLASS__"]

// Training annotations = area annotations carrying one of the two training classes.
// Each detection whose centroid lies inside one gets that annotation's class as its ground-truth label
// (this mirrors how QuPath's "Train object classifier" uses annotations).
def trainAnnots = getAnnotationObjects().findAll {
    !it.getROI().isLine() && labelClasses.contains(it.getPathClass()?.toString())
}
if (trainAnnots.isEmpty()) {
    println "TRAINING_EXPORT_NONE " + imageName + " (no training annotations)"
    return
}

def rows = []
getDetectionObjects().each { d ->
    double cx = d.getROI().getCentroidX()
    double cy = d.getROI().getCentroidY()
    def hit = trainAnnots.find { a -> a.getROI().contains(cx, cy) }
    if (hit == null) return
    def ml = d.getMeasurementList()
    def row = [imageName, d.getID().toString(), hit.getPathClass().toString(),
               d.getPathClass()?.toString() ?: "Unclassified"]
    features.each { f -> row << (ml.containsKey(f) ? ml.get(f).toString() : "") }
    rows << row
}

def outDir = buildFilePath(PROJECT_BASE_DIR, "training_data")
mkdirs(outDir)
def outPath = buildFilePath(outDir, imageName.replaceAll("[^a-zA-Z0-9.-]", "_") + "_training.tsv")
new File(outPath).withWriter { w ->
    w.writeLine((["Image", "ObjectID", "Label", "Predicted"] + features).join("\t"))
    rows.each { w.writeLine(it.join("\t")) }
}
println "Exported " + rows.size() + " labelled detections to " + outPath
println "TRAINING_EXPORT_OK " + imageName
"""

def build_pipeline_script(run_classifier=True):
    s = PIPELINE_TEMPLATE
    for k, v in {
        "__CLASSIFIER_NAME__": CLASSIFIER_NAME, "__CELLPOSE_MODEL__": CELLPOSE_MODEL,
        "__PIXEL_SIZE__": PIXEL_SIZE_UM, "__DIAMETER__": CELL_DIAMETER, "__EXPANSION__": CELL_EXPANSION,
        "__RGB_THRESHOLD__": NM_RGB_THRESHOLD, "__RUN_CLASSIFIER__": str(run_classifier).lower(),
        "__POS_CLASS__": POS_CLASS,
    }.items():
        s = s.replace(k, str(v))
    return s

def build_training_script():
    s = TRAINING_EXPORT_TEMPLATE
    for k, v in {"__CLASSIFIER_NAME__": CLASSIFIER_NAME, "__POS_CLASS__": POS_CLASS, "__NEG_CLASS__": NEG_CLASS}.items():
        s = s.replace(k, v)
    return s

pipeline_script_path = scripts_dir / "full_pipeline.groovy"
pipeline_script_path.write_text(build_pipeline_script(run_classifier=True))
training_script_path = scripts_dir / "export_training_data.groovy"
training_script_path.write_text(build_training_script())
print("Wrote:", pipeline_script_path)
print("Wrote:", training_script_path)

Wrote: /Users/seymour/Downloads/Shomaiya/Neuromelanin/qupath/scripts/full_pipeline.groovy
Wrote: /Users/seymour/Downloads/Shomaiya/Neuromelanin/qupath/scripts/export_training_data.groovy


## 4. Run the full pipeline headlessly

For each slide this launches QuPath with no GUI: Cellpose (if no detections yet) → NM measurements → preflight → classifier → counts → per-cell NM export. A slide only counts as successful if the script printed its `PIPELINE_OK` marker (QuPath's exit code alone isn't reliable). Full output per slide is saved in `pipeline_logs/`.

Run slides **sequentially** — two QuPath processes writing to the same project can clash.

In [4]:
def run_qupath(script_path, image_name, tag):
    cmd = [QUPATH_EXE, "script", str(script_path),
           "--project", str(PROJECT_PATH), "--image", image_name]
    t0 = time.time()
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=TIMEOUT_SEC)
        out, err, rc = r.stdout, r.stderr, r.returncode
    except subprocess.TimeoutExpired as e:
        out, err, rc = (e.stdout or ""), f"TIMEOUT after {TIMEOUT_SEC}s", -1
    log = logs_dir / f"{tag}_{safe_name(image_name)}.log"
    log.write_text(out + "\n--- STDERR ---\n" + err)
    return {"image": image_name, "returncode": rc, "seconds": round(time.time() - t0),
            "combined": out + err, "log": str(log)}

def run_stage(script_path, ok_marker, tag, images=None):
    rows = []
    for name in (images or IMAGE_NAMES):
        print(f"[{tag}] {name} ...", flush=True)
        res = run_qupath(script_path, name, tag)
        ok = res["returncode"] == 0 and ok_marker in res["combined"]
        status = "OK" if ok else ("PREFLIGHT_FAILED" if "PREFLIGHT_FAILED" in res["combined"]
                 else "SKIPPED" if "PIPELINE_SKIPPED" in res["combined"] else "FAILED")
        rows.append({"image": name, "status": status, "returncode": res["returncode"],
                     "seconds": res["seconds"], "log": res["log"]})
        if status != "OK":
            print("   ->", status, "- see", res["log"])
            print("   ", res["combined"][-1200:].replace("\n", "\n    "))
    return pd.DataFrame(rows)

# regenerate in case parameters above changed
pipeline_script_path.write_text(build_pipeline_script(run_classifier=True))
run_log = run_stage(pipeline_script_path, "PIPELINE_OK", "pipeline")
run_log

[pipeline] NPBB8_HE_B11.svs ...


PermissionError: [Errno 13] Permission denied: '/Applications/QuPath-0.7.0-arm64.app'

In [1]:
import os
print(repr(QUPATH_EXE))
print(os.path.isdir(QUPATH_EXE), os.path.isfile(QUPATH_EXE), os.access(QUPATH_EXE, os.X_OK))

NameError: name 'QUPATH_EXE' is not defined

## 5. Collect NM measurements and slide summary

In [ ]:
tsv_files = sorted(nm_dir.glob("*_neuromelanin.tsv"))
print(f"{len(tsv_files)} result files")
df_all = pd.concat([pd.read_csv(f, sep="\t") for f in tsv_files], ignore_index=True)
NM_COLS = [c for c in df_all.columns if c.startswith("NM:")]

# --- exports ---
df_all.to_csv(nm_dir / "ALL_cells_neuromelanin.csv", index=False)
df_pig = df_all[df_all["Class"] == POS_CLASS].copy()
df_pig.to_csv(nm_dir / "PIGMENTED_neurons_neuromelanin.csv", index=False)

# --- per-slide counts ---
counts = df_all.groupby("Image")["Class"].value_counts().unstack(fill_value=0)
for c in (POS_CLASS, NEG_CLASS):
    if c not in counts: counts[c] = 0
counts["Total detections"] = counts.sum(axis=1)
counts["% Pigmented (of Pigmented+Negative)"] = counts[POS_CLASS] / (counts[POS_CLASS] + counts[NEG_CLASS]) * 100

# --- per-slide NM burden, pigmented neurons only ---
burden = df_pig.groupby("Image")[NM_COLS].mean().add_prefix("mean ")
summary = counts.join(burden)
summary.to_csv(nm_dir / "slide_summary.csv")
print("Saved:", nm_dir / "ALL_cells_neuromelanin.csv")
print("Saved:", nm_dir / "PIGMENTED_neurons_neuromelanin.csv")
print("Saved:", nm_dir / "slide_summary.csv")
summary

## 6. Training metrics

**Prerequisite:** the slides contain your training annotations (area annotations classed `Pigmented Neuron` / `Negative`) and have been through the pipeline above (so the detections carry all classifier features).

Three views of performance:

| View | What it tells you | Caveat |
|---|---|---|
| **QuPath classifier on its own training cells** | How well the saved classifier reproduces your labels | Optimistic — it was trained on these cells |
| **Stratified k-fold CV** (scikit-learn Random Forest, same hyper-parameters) | Honest estimate for unseen cells from the same slides | Proxy model — not the exact OpenCV RTrees model, but the same family and settings |
| **Leave-slides-out CV** (grouped by image) | How well it should generalise to a *new slide* | Needs ≥2 slides with labels; usually the lowest, and the most relevant for your PD vs control comparisons |

In [ ]:
# 6a. snapshot the current classifier (so metrics are tied to a specific version)
clf_hash = hashlib.sha256(classifier_path.read_bytes()).hexdigest()[:12]
snap = versions_dir / f"{CLASSIFIER_NAME}_{clf_hash}.json"
if not snap.exists():
    shutil.copy(classifier_path, snap)
print("Classifier version:", clf_hash, "| QuPath timestamp:", clf_json.get("timestamp"))

# 6b. export labelled training cells from every slide
train_log = run_stage(training_script_path, "TRAINING_EXPORT_OK", "train_export")
train_log

In [ ]:
train_files = sorted(train_dir.glob("*_training.tsv"))
assert train_files, "No training data exported - check that slides contain Pigmented Neuron / Negative annotations."
td = pd.concat([pd.read_csv(f, sep="\t") for f in train_files], ignore_index=True)
td = td[td["Label"].isin([POS_CLASS, NEG_CLASS])].copy()

missing_cols = [f for f in FEATURE_NAMES if f not in td.columns]
assert not missing_cols, f"Missing feature columns: {missing_cols}"
n_nan = td[FEATURE_NAMES].isna().sum().sum()
if n_nan: print(f"WARNING: {n_nan} blank feature values - filled with 0 (the classifier's missing_subst)")

X = td[FEATURE_NAMES].fillna(0).values
y = (td["Label"] == POS_CLASS).astype(int).values
groups = td["Image"].values

print(f"{len(td)} labelled cells across {td['Image'].nunique()} slides")
print(td["Label"].value_counts().to_string())
print("\nPer slide:")
print(td.groupby(["Image", "Label"]).size().unstack(fill_value=0).to_string())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, balanced_accuracy_score)

def metrics_from(y_true, y_pred, y_score=None):
    m = {"n": int(len(y_true)),
         "accuracy": accuracy_score(y_true, y_pred),
         "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
         "precision_pigmented": precision_score(y_true, y_pred, zero_division=0),
         "recall_pigmented": recall_score(y_true, y_pred, zero_division=0),
         "f1_pigmented": f1_score(y_true, y_pred, zero_division=0)}
    if y_score is not None and len(set(y_true)) == 2:
        m["roc_auc"] = roc_auc_score(y_true, y_score)
    m["confusion_matrix [[TN, FP], [FN, TP]]"] = confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist()
    return m

# match the QuPath classifier's settings (ntrees 50, max depth 25, min sample count 10)
st = clf_json["classifier"]["statmodel"]["opencv_ml_rtrees"]
N_TREES   = st["ntrees"]
MAX_DEPTH = st["training_params"]["max_depth"]
MIN_SPLIT = st["training_params"]["min_sample_count"]
rf = RandomForestClassifier(n_estimators=N_TREES, max_depth=MAX_DEPTH, min_samples_split=MIN_SPLIT,
                            random_state=0, n_jobs=-1)

results, cms = {}, {}

# (1) QuPath classifier vs. your labels (resubstitution)
pred_qp = td["Predicted"]
has_pred = pred_qp.isin([POS_CLASS, NEG_CLASS])
if has_pred.any():
    yq_true = y[has_pred.values]
    yq_pred = (pred_qp[has_pred] == POS_CLASS).astype(int).values
    results["QuPath classifier (on its training cells)"] = metrics_from(yq_true, yq_pred)
    if (~has_pred).any():
        print(f"Note: {(~has_pred).sum()} labelled cells had no class from the classifier - re-run section 4 first.")
else:
    print("No classifier predictions found on the labelled cells - run section 4 first.")

# (2) stratified k-fold CV
n_pos, n_neg = y.sum(), (1 - y).sum()
k = int(min(5, n_pos, n_neg))
if k >= 2:
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=0)
    proba = cross_val_predict(rf, X, y, cv=skf, method="predict_proba")[:, 1]
    results[f"Stratified {k}-fold CV (sklearn RF proxy)"] = metrics_from(y, (proba >= 0.5).astype(int), proba)

# (3) leave-slides-out CV
n_groups = len(set(groups))
if n_groups >= 2:
    gk = GroupKFold(n_splits=min(5, n_groups))
    proba_g = cross_val_predict(rf, X, y, cv=gk, groups=groups, method="predict_proba")[:, 1]
    results[f"Leave-slides-out CV, {min(5, n_groups)} folds (sklearn RF proxy)"] = metrics_from(y, (proba_g >= 0.5).astype(int), proba_g)
else:
    print("Only one labelled slide - leave-slides-out CV skipped.")

scalar_rows = {name: {k_: v for k_, v in m.items() if not k_.startswith("confusion")} for name, m in results.items()}
metrics_table = pd.DataFrame(scalar_rows).T
display(metrics_table.round(3))

In [ ]:
# confusion matrices + feature importance
fig, axes = plt.subplots(1, len(results) + 1, figsize=(4.2 * (len(results) + 1), 4))
axes = np.atleast_1d(axes)
for ax, (name, m) in zip(axes, results.items()):
    cm = np.array(m["confusion_matrix [[TN, FP], [FN, TP]]"])
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1], [NEG_CLASS, POS_CLASS], rotation=20); ax.set_yticks([0, 1], [NEG_CLASS, POS_CLASS])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Your label")
    ax.set_title(name.split(" (")[0], fontsize=9)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")

rf.fit(X, y)
imp = pd.Series(rf.feature_importances_, index=FEATURE_NAMES).sort_values()
axes[-1].barh(imp.index, imp.values); axes[-1].set_title("Feature importance (sklearn proxy)", fontsize=9)
axes[-1].tick_params(axis="y", labelsize=7)
plt.tight_layout()
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
fig.savefig(metrics_dir / f"training_metrics_{stamp}.png", dpi=200)
plt.show()

# save metrics, tied to the classifier version
out = {"timestamp": stamp, "classifier": CLASSIFIER_NAME, "classifier_sha256_12": clf_hash,
       "n_labelled_cells": int(len(td)), "n_slides": int(n_groups),
       "class_counts": td["Label"].value_counts().to_dict(),
       "rf_proxy_settings": {"n_estimators": N_TREES, "max_depth": MAX_DEPTH, "min_samples_split": MIN_SPLIT},
       "results": results}
(metrics_dir / f"training_metrics_{stamp}.json").write_text(json.dumps(out, indent=2, default=float))
metrics_table.round(4).to_csv(metrics_dir / f"training_metrics_{stamp}.csv")
td.to_csv(metrics_dir / f"training_table_{stamp}.csv", index=False)
print("Saved metrics to", metrics_dir)

## 7. Retraining the classifier

**Yes — retraining is done manually in the QuPath GUI.** Creating the labelled cells (clicking/annotating which are Pigmented vs Negative) is interactive by nature, and QuPath's training dialog is the only supported way to produce a classifier that `runObjectClassifier` can load. The notebook handles everything before and after.

**Retrain loop**
1. Run sections 4–5 here first, so every detection already carries all 21 features (including the `NM:` ones).
2. Open the project in QuPath. On the slides you want, add/fix training annotations classed `Pigmented Neuron` and `Negative` (the cells they contain become training examples).
3. *Classify → Object classification → Train object classifier*:
   - Object filter: **Detections (all)**
   - Classifier: **Random trees**; open *Advanced options* and keep (or deliberately change) the current settings: 50 trees, max depth 25, min sample count 10
   - Features: the same 21 measurements as before (the list printed in section 2)
   - Classes: `Pigmented Neuron`, `Negative`; Training: **Annotations**
   - **Save under the same name** (`Pigmented_Neurons`) so the pipeline picks it up automatically.
4. Back here: re-run section 2 (reloads the classifier + feature list), then 4 → 5 → 6. Section 6 snapshots each classifier version and stores its metrics next to its hash in `training_metrics/`, so you can compare versions.

If you ever change the feature list in the GUI, section 2 picks the new list up automatically from the saved JSON, and the preflight check will tell you if any new feature isn't being computed for the detections.